In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

DATA_ROOT = Path("wandb_visual_benchmark_downloads") / "abstract_visual"

RUNS = {
    "Qwen2.5-VL-3B": DATA_ROOT / "abstract_visual_qwen25vl_3b",
    "Qwen2.5-VL-7B": DATA_ROOT / "abstract_visual_qwen25vl_7b",
}

OUT_DIR = DATA_ROOT / "plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CHART_TYPES = ["bar", "slider", "dot", "grid", "line", "scatter"]
TASK_TYPES  = [
    "value_at_step", "argmax_at_step", "comparison_at_step",
    "changed_variable", "change_direction", "change_amount",
]

for model, path in RUNS.items():
    ok_csv = (path / "results.csv").exists()
    ok_agg = (path / "aggregate.json").exists()
    print(f"{model}: results.csv={ok_csv}  aggregate.json={ok_agg}")

In [ ]:
frames     = []
aggregates = {}

for model_name, run_dir in RUNS.items():
    df = pd.read_csv(run_dir / "results.csv")
    df["model"] = model_name

    df["base_chart"] = df["modality"].str.replace(r"_(ocr|nocr)$", "", regex=True)
    df["ocr_condition"] = df["modality"].apply(
        lambda m: "ocr" if m.endswith("_ocr") else ("nocr" if m.endswith("_nocr") else "none")
    )

    df["correct"]       = df["correct"].astype(str).str.lower().isin(["true", "1", "yes"])
    df["numeric_error"] = pd.to_numeric(df["numeric_error"], errors="coerce")

    frames.append(df)

    with open(run_dir / "aggregate.json") as f:
        aggregates[model_name] = json.load(f)

all_df = pd.concat(frames, ignore_index=True)
print(f"Rows: {len(all_df)}")
print(f"Modalities : {sorted(all_df['modality'].unique())}")
print(f"Task types : {sorted(all_df['task_type'].unique())}")
display(all_df[["model", "modality", "base_chart", "ocr_condition", "task_type", "correct"]].head(8))

In [ ]:
# 1. Overall accuracy
overall = all_df.groupby("model")["correct"].mean().reset_index(name="accuracy")
display(overall.assign(accuracy=overall["accuracy"].map("{:.1%}".format)))

fig, ax = plt.subplots(figsize=(max(4, len(RUNS) * 2 + 1), 4))
bars = ax.bar(overall["model"], overall["accuracy"] * 100)
ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.set_ylim(0, 110)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylabel("Accuracy")
ax.set_title("Overall Accuracy")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUT_DIR / "01_overall_accuracy.png", dpi=200)
plt.show()

In [ ]:
# 2. Accuracy by base chart type (OCR + no-OCR combined)
chart_acc = (
    all_df.groupby(["model", "base_chart"])["correct"]
    .mean()
    .reset_index(name="accuracy")
)
pivot_chart = (
    chart_acc.pivot(index="base_chart", columns="model", values="accuracy")
    .reindex([c for c in CHART_TYPES if c in all_df["base_chart"].unique()])
)
display(pivot_chart.map(lambda x: f"{x:.1%}" if pd.notna(x) else "-"))

ax = (pivot_chart * 100).plot(kind="bar", figsize=(10, 5))
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 110)
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy by Chart Type (OCR + no-OCR combined)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(OUT_DIR / "02_accuracy_by_chart_type.png", dpi=200)
plt.show()

In [ ]:
# 3. OCR vs no-OCR side-by-side per chart type
ocr_df = all_df[all_df["ocr_condition"].isin(["ocr", "nocr"])]
ocr_acc = (
    ocr_df.groupby(["model", "base_chart", "ocr_condition"])["correct"]
    .mean()
    .reset_index(name="accuracy")
)

valid_charts = [c for c in CHART_TYPES if c in all_df["base_chart"].unique()]
models = list(RUNS.keys())
n_models = len(models)

fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5), sharey=True)
if n_models == 1:
    axes = [axes]

x = np.arange(len(valid_charts))
w = 0.35

for ax, model in zip(axes, models):
    sub = ocr_acc[ocr_acc["model"] == model]

    def _val(chart, cond):
        row = sub[(sub["base_chart"] == chart) & (sub["ocr_condition"] == cond)]
        return row["accuracy"].values[0] * 100 if len(row) else np.nan

    ocr_vals  = [_val(c, "ocr")  for c in valid_charts]
    nocr_vals = [_val(c, "nocr") for c in valid_charts]

    ax.bar(x - w / 2, ocr_vals,  w, label="OCR")
    ax.bar(x + w / 2, nocr_vals, w, label="no-OCR")
    ax.set_xticks(x)
    ax.set_xticklabels(valid_charts, rotation=25, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylim(0, 110)
    ax.set_title(model)
    ax.legend()

fig.suptitle("OCR vs no-OCR Accuracy by Chart Type")
plt.tight_layout()
plt.savefig(OUT_DIR / "03_ocr_vs_nocr_by_chart.png", dpi=200)
plt.show()

In [ ]:
# 4. OCR Gain
gain_rows = []
for model, agg in aggregates.items():
    for row in agg.get("ocr_gain", []):
        gain_rows.append({"model": model, **row})

gain_df = pd.DataFrame(gain_rows)
display(
    gain_df[["model", "base_modality", "n_ocr", "n_nocr", "ocr_accuracy", "nocr_accuracy", "ocr_gain"]]
    .assign(
        ocr_accuracy  = gain_df["ocr_accuracy"].map("{:.1%}".format),
        nocr_accuracy = gain_df["nocr_accuracy"].map("{:.1%}".format),
        ocr_gain      = gain_df["ocr_gain"].map("{:+.1%}".format),
    )
)

pivot_gain = (
    gain_df.pivot(index="base_modality", columns="model", values="ocr_gain")
    .reindex([c for c in CHART_TYPES if c in gain_df["base_modality"].unique()])
)

fig, ax = plt.subplots(figsize=(9, 5))
(pivot_gain * 100).plot(kind="bar", ax=ax)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylabel("OCR Gain (pp)")
ax.set_title("OCR Gain = Acc(OCR) - Acc(no-OCR) by Chart Type")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(OUT_DIR / "04_ocr_gain.png", dpi=200)
plt.show()

In [ ]:
# 5. Accuracy by task type
task_acc = (
    all_df.groupby(["model", "task_type"])["correct"]
    .mean()
    .reset_index(name="accuracy")
)
pivot_task = (
    task_acc.pivot(index="task_type", columns="model", values="accuracy")
    .reindex([t for t in TASK_TYPES if t in all_df["task_type"].unique()])
)
display(pivot_task.map(lambda x: f"{x:.1%}" if pd.notna(x) else "-"))

ax = (pivot_task * 100).plot(kind="bar", figsize=(11, 5))
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 110)
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy by Task Type")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUT_DIR / "05_accuracy_by_task_type.png", dpi=200)
plt.show()

In [ ]:
# 6. Acc++（按 (pair_id, modality) 分组，从 results.csv 重新计算）
# 注意：aggregate.json 里的 Acc++ = 0 是因为旧版按 pair_id 分组，
# 把 12 个 modality × 2 polarity = 24 个 case 放进同一个 pair，
# 要求全部 24 个都对才算这对 pair 正确，几乎不可能。
# 正确做法：按 (pair_id, modality) 分组，每个 pair 只含 pos + neg 两个 case。

paired_all = all_df[all_df["pair_id"].notna()].copy()

pair_correct_all = (
    paired_all.groupby(["model", "pair_id", "modality"])["correct"]
    .agg(pair_correct=lambda s: s.all())
    .reset_index()
)

accpp_overall = (
    pair_correct_all.groupby("model")["pair_correct"]
    .agg(n_pairs="count", acc_pp="mean")
    .reset_index()
)

overall_acc = all_df.groupby("model")["correct"].mean().reset_index(name="accuracy")
compare = overall_acc.merge(accpp_overall, on="model")
display(
    compare.assign(
        accuracy = compare["accuracy"].map("{:.1%}".format),
        acc_pp   = compare["acc_pp"].map("{:.1%}".format),
    )
)

x = np.arange(len(compare))
w = 0.35
fig, ax = plt.subplots(figsize=(max(4, len(RUNS) * 2 + 1), 4))
b1 = ax.bar(x - w / 2, compare["accuracy"] * 100, w, label="Accuracy")
b2 = ax.bar(x + w / 2, compare["acc_pp"]   * 100, w, label="Acc++ (per modality)")
ax.bar_label(b1, fmt="%.1f%%", padding=3, fontsize=8)
ax.bar_label(b2, fmt="%.1f%%", padding=3, fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(compare["model"], rotation=20, ha="right")
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 120)
ax.set_ylabel("Score")
ax.set_title("Accuracy vs Acc++ (grouped by pair_id × modality)")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "06_acc_vs_accpp.png", dpi=200)
plt.show()

# Acc++ by modality
pair_correct_all["base_chart"] = pair_correct_all["modality"].str.replace(r"_(ocr|nocr)$", "", regex=True)
pivot_pp = (
    pair_correct_all.groupby(["model", "modality"])["pair_correct"]
    .mean()
    .reset_index(name="acc_pp")
    .pivot(index="modality", columns="model", values="acc_pp")
)
display(pivot_pp.map(lambda x: f"{x:.1%}" if pd.notna(x) else "-"))

In [ ]:
# 7. Heatmap: base_chart x task_type
valid_tasks  = [t for t in TASK_TYPES  if t in all_df["task_type"].unique()]
valid_charts = [c for c in CHART_TYPES if c in all_df["base_chart"].unique()]

for model in models:
    sub  = all_df[all_df["model"] == model]
    heat = (
        sub.groupby(["base_chart", "task_type"])["correct"]
        .mean()
        .unstack(fill_value=np.nan)
        .reindex(index=valid_charts, columns=valid_tasks)
    )

    fig, ax = plt.subplots(figsize=(max(8, len(valid_tasks) * 1.6), 4.5))
    im = ax.imshow(heat.values * 100, aspect="auto", vmin=0, vmax=100, cmap="RdYlGn")
    ax.set_xticks(range(len(valid_tasks)))
    ax.set_yticks(range(len(valid_charts)))
    ax.set_xticklabels(valid_tasks, rotation=30, ha="right", fontsize=9)
    ax.set_yticklabels(valid_charts)
    for i in range(len(valid_charts)):
        for j in range(len(valid_tasks)):
            v = heat.iloc[i, j]
            if pd.notna(v):
                color = "white" if v < 0.35 or v > 0.75 else "black"
                ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=9, color=color)
    plt.colorbar(im, ax=ax, label="Accuracy")
    ax.set_title(f"{model}: Chart Type x Task Type")
    plt.tight_layout()
    safe = model.replace("/", "_").replace(".", "")
    plt.savefig(OUT_DIR / f"07_heatmap_chart_task_{safe}.png", dpi=200)
    plt.show()

In [ ]:
# 8. Heatmap: base_chart x ocr_condition
for model in models:
    sub  = all_df[(all_df["model"] == model) & (all_df["ocr_condition"].isin(["ocr", "nocr"]))]
    heat = (
        sub.groupby(["base_chart", "ocr_condition"])["correct"]
        .mean()
        .unstack(fill_value=np.nan)
        .reindex(index=valid_charts, columns=["ocr", "nocr"])
    )
    heat["gain"] = heat["ocr"] - heat["nocr"]
    display(f"== {model} ==")
    display(
        heat.map(lambda x: f"{x:+.1%}" if pd.notna(x) else "-")
        .rename(columns={"ocr": "Acc (OCR)", "nocr": "Acc (no-OCR)", "gain": "OCR Gain"})
    )

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(heat[["ocr", "nocr"]].values * 100, aspect="auto", vmin=0, vmax=100, cmap="RdYlGn")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["OCR", "no-OCR"])
    ax.set_yticks(range(len(valid_charts)))
    ax.set_yticklabels(valid_charts)
    for i, chart in enumerate(valid_charts):
        for j, cond in enumerate(["ocr", "nocr"]):
            v = heat.loc[chart, cond] if chart in heat.index else np.nan
            if pd.notna(v):
                color = "white" if v < 0.35 or v > 0.75 else "black"
                ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=9, color=color)
    plt.colorbar(im, ax=ax, label="Accuracy")
    ax.set_title(f"{model}: OCR vs no-OCR")
    plt.tight_layout()
    safe = model.replace("/", "_").replace(".", "")
    plt.savefig(OUT_DIR / f"08_heatmap_ocr_nocr_{safe}.png", dpi=200)
    plt.show()

In [ ]:
# 9. Numeric error analysis
num_df = all_df[all_df["numeric_error"].notna()].copy()

err_summary = (
    num_df.groupby(["model", "base_chart", "task_type"])["numeric_error"]
    .agg(n="count", median="median", p95=lambda x: x.quantile(0.95), mean="mean")
    .reset_index()
)
display(err_summary)

pivot_err = (
    num_df.groupby(["model", "base_chart"])["numeric_error"]
    .median()
    .reset_index(name="median_error")
    .pivot(index="base_chart", columns="model", values="median_error")
    .reindex(valid_charts)
)

ax = pivot_err.plot(kind="bar", figsize=(9, 4))
ax.set_ylabel("Median Absolute Error")
ax.set_title("Median Numeric Error by Chart Type")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(OUT_DIR / "09_median_numeric_error.png", dpi=200)
plt.show()

In [ ]:
# ── 10. changed_variable 污染分析 ────────────────────────────────────────────
# 诊断：模型答了什么，为什么全错

cv = all_df[all_df["task_type"] == "changed_variable"].copy()

direction_words  = {"increase", "decrease", "same"}
color_words      = {"blue", "green", "red", "orange", "purple", "yellow"}
allowed_vars     = {"a", "b", "c"}

def _norm(x):
    return str(x).strip().lower() if pd.notna(x) else ""

cv["pred_norm"] = cv["parsed_answer"].apply(_norm)

cv["answer_group"] = np.select(
    [
        cv["pred_norm"].isin(allowed_vars),
        cv["pred_norm"].isin(direction_words),
        cv["pred_norm"].isin(color_words),
    ],
    ["valid_variable_name", "direction_word", "color_word"],
    default="other",
)

contam = (
    cv.groupby(["model", "answer_group"])
    .size()
    .groupby(level=0)
    .transform(lambda x: x / x.sum())
    .reset_index(name="fraction")
    .pivot(index="model", columns="answer_group", values="fraction")
    .fillna(0)
)

print("=== changed_variable: answer distribution ===")
print("(direction_word → prompt instruction contamination)")
display(contam.map(lambda x: f"{x:.1%}"))
contam.mul(100).plot(kind="bar", figsize=(8, 4))
plt.ylabel("Fraction of answers (%)")
plt.title("changed_variable: What did the model actually answer?")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUT_DIR / "10_cv_contamination.png", dpi=200)
plt.show()

print("\n=== Sample rows ===")
display(cv[["model","modality","gold_answer","parsed_answer","answer_group","raw_output"]].head(20))

In [ ]:
# ── 11. Clean accuracy（排除 changed_variable 及其 assertion）────────────────
# changed_variable 因 prompt instruction contamination 全部答错（0%），
# 排除后才能反映模型真实的视觉理解能力。

EXCLUDE_TASKS = {"changed_variable", "changed_variable_assertion"}

df_clean = all_df[~all_df["task_type"].isin(EXCLUDE_TASKS)].copy()

# 总体对比
overall_raw   = all_df.groupby("model")["correct"].mean().rename("raw_accuracy")
overall_clean = df_clean.groupby("model")["correct"].mean().rename("clean_accuracy")
summary = pd.concat([overall_raw, overall_clean], axis=1)
summary["delta"] = summary["clean_accuracy"] - summary["raw_accuracy"]
display(summary.map(lambda x: f"{x:+.1%}" if pd.notna(x) else "-")
        .assign(delta=summary["delta"].map("{:+.1%}".format)))

# 并排柱状图
x = np.arange(len(summary))
w = 0.35
fig, ax = plt.subplots(figsize=(max(4, len(RUNS) * 2 + 1), 4))
b1 = ax.bar(x - w/2, summary["raw_accuracy"]   * 100, w, label="All tasks (raw)")
b2 = ax.bar(x + w/2, summary["clean_accuracy"] * 100, w, label="Excl. changed_variable")
ax.bar_label(b1, fmt="%.1f%%", padding=3, fontsize=8)
ax.bar_label(b2, fmt="%.1f%%", padding=3, fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(summary.index, rotation=20, ha="right")
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 120)
ax.set_ylabel("Accuracy")
ax.set_title("Overall Accuracy: raw vs. excluding changed_variable")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "11_clean_vs_raw_overall.png", dpi=200)
plt.show()

In [ ]:
# ── 12. Clean task accuracy（按 task type，排除 changed_variable）──────────────
CLEAN_TASK_TYPES = [t for t in TASK_TYPES if t not in EXCLUDE_TASKS]

clean_task = (
    df_clean.groupby(["model", "task_type"])["correct"]
    .mean()
    .reset_index(name="accuracy")
)
pivot_clean_task = (
    clean_task.pivot(index="task_type", columns="model", values="accuracy")
    .reindex([t for t in CLEAN_TASK_TYPES if t in df_clean["task_type"].unique()])
)
display(pivot_clean_task.map(lambda x: f"{x:.1%}" if pd.notna(x) else "-"))

ax = (pivot_clean_task * 100).plot(kind="bar", figsize=(11, 5))
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 110)
ax.set_ylabel("Accuracy")
ax.set_title("Clean Task Accuracy (changed_variable excluded)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUT_DIR / "12_clean_accuracy_by_task.png", dpi=200)
plt.show()

In [ ]:
# ── 13. Clean heatmap: chart type × task type（排除 changed_variable）─────────
clean_valid_tasks  = [t for t in CLEAN_TASK_TYPES if t in df_clean["task_type"].unique()]
clean_valid_charts = [c for c in CHART_TYPES if c in df_clean["base_chart"].unique()]

for model in models:
    sub  = df_clean[df_clean["model"] == model]
    heat = (
        sub.groupby(["base_chart", "task_type"])["correct"]
        .mean()
        .unstack(fill_value=np.nan)
        .reindex(index=clean_valid_charts, columns=clean_valid_tasks)
    )

    fig, ax = plt.subplots(figsize=(max(8, len(clean_valid_tasks) * 1.8), 4.5))
    im = ax.imshow(heat.values * 100, aspect="auto", vmin=0, vmax=100, cmap="RdYlGn")
    ax.set_xticks(range(len(clean_valid_tasks)))
    ax.set_yticks(range(len(clean_valid_charts)))
    ax.set_xticklabels(clean_valid_tasks, rotation=30, ha="right", fontsize=9)
    ax.set_yticklabels(clean_valid_charts)
    for i in range(len(clean_valid_charts)):
        for j in range(len(clean_valid_tasks)):
            v = heat.iloc[i, j]
            if pd.notna(v):
                color = "white" if v < 0.35 or v > 0.75 else "black"
                ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=9, color=color)
    plt.colorbar(im, ax=ax, label="Accuracy")
    ax.set_title(f"{model}: Chart Type x Task Type  (changed_variable excluded)")
    plt.tight_layout()
    safe = model.replace("/", "_").replace(".", "")
    plt.savefig(OUT_DIR / f"13_clean_heatmap_{safe}.png", dpi=200)
    plt.show()

In [ ]:
# ── 14. Clean Acc++（排除 changed_variable，按 pair_id × modality 分组）────────
paired_clean = df_clean[df_clean["pair_id"].notna()].copy()

pair_correct_clean = (
    paired_clean.groupby(["model", "pair_id", "modality"])["correct"]
    .agg(pair_correct=lambda s: s.all())
    .reset_index()
)

clean_accpp = (
    pair_correct_clean.groupby("model")["pair_correct"]
    .mean()
    .rename("clean_acc_pp")
)

raw_accpp = (
    pair_correct_all.groupby("model")["pair_correct"]
    .mean()
    .rename("raw_acc_pp")
)

accpp_compare = pd.concat([
    all_df.groupby("model")["correct"].mean().rename("raw_accuracy"),
    df_clean.groupby("model")["correct"].mean().rename("clean_accuracy"),
    raw_accpp,
    clean_accpp,
], axis=1)
accpp_compare["accpp_delta"] = accpp_compare["clean_acc_pp"] - accpp_compare["raw_acc_pp"]

display(accpp_compare.map(lambda x: f"{x:.1%}" if pd.notna(x) else "-")
        .rename(columns={
            "raw_accuracy":   "Acc (all)",
            "clean_accuracy": "Acc (excl. CV)",
            "raw_acc_pp":     "Acc++ (all)",
            "clean_acc_pp":   "Acc++ (excl. CV)",
            "accpp_delta":    "Acc++ delta",
        }))

x = np.arange(len(accpp_compare))
w = 0.2
fig, ax = plt.subplots(figsize=(max(5, len(RUNS) * 3), 5))
for i, (col, label) in enumerate([
    ("raw_accuracy",   "Acc (all tasks)"),
    ("clean_accuracy", "Acc (excl. CV)"),
    ("raw_acc_pp",     "Acc++ (all tasks)"),
    ("clean_acc_pp",   "Acc++ (excl. CV)"),
]):
    bars = ax.bar(x + (i - 1.5) * w, accpp_compare[col] * 100, w, label=label)
    ax.bar_label(bars, fmt="%.1f%%", padding=2, fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(accpp_compare.index, rotation=20, ha="right")
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 120)
ax.set_ylabel("Score")
ax.set_title("Accuracy & Acc++: raw vs. excluding changed_variable (CV)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "14_clean_accpp_comparison.png", dpi=200)
plt.show()